In [ ]:
(* load the setup of the notebooks *)
#use "../setup.ml";;

# The Set $\mathbb{Q}$ is Countable

OCaml has no built-in type for rational numbers.  However, the library [Zarith](https://github.com/ocaml/Zarith) provides the module `Q`, which implements rational numbers.  A value of type `Q.t` is a fraction $\frac{p}{q}$ where both $p$ and $q$ are arbitrary large integers.  Fractions are always stored in reduced form.  The directive `#require "zarith.top"` loads Zarith together with the *printers* that display its numbers in the toplevel.

In [ ]:
#require "zarith.top";;

The function call `Q.of_ints p q` creates the fraction $\frac{p}{q}$ from the integers `p` and `q`.  Inside the *local open* `Q.(...)`, the names defined in the module `Q` can be used without the prefix `Q.`.  In particular, the operators `+`, `-`, `*`, and `/` denote the arithmetic operations on rational numbers.

In [ ]:
let result = Q.(of_ints (-1) 2 + of_ints 1 3)

The function `Q.to_string` converts a rational number into a string.

In [ ]:
Q.to_string result

The function `gcd` takes two arguments:
- `a` is a natural number,
- `b` is a natural number.

It returns the *greatest common divisor* of `a` and `b`.

**Implementation:** The function implements the *Euclidean algorithm*, which is based on the following two equations:
- $\texttt{gcd}(a, 0) = a$,
- $\texttt{gcd}(a, b) = \texttt{gcd}(b, a \;\texttt{mod}\; b)$ if $b \not= 0$.

In [ ]:
let rec gcd (a : int) (b : int) : int =
  if b = 0 then
    a
  else
    gcd b (a mod b)

In [ ]:
gcd 120 100

We enumerate the rational numbers by the sum of their numerator and their denominator.  The function `fractions_with_sum` takes one argument:
- `sum` is a natural number such that $\texttt{sum} \geq 2$.

It returns the sequence of all fractions $\frac{p}{q}$ such that $p > 0$, $q > 0$, $p + q = \texttt{sum}$, and $\texttt{gcd}(p, q) = 1$.  This sequence is finite.  Sequences are explained in the notebook `Primes.ipynb`.  The condition $\texttt{gcd}(p, q) = 1$ ensures that every rational number is enumerated exactly once.

**Implementation:**
- `Seq.init (sum - 1) (fun i -> i + 1)` is the sequence $1, 2, \cdots, \texttt{sum} - 1$ of the possible numerators.
- `Seq.filter` keeps those numerators that are coprime with the corresponding denominator $\texttt{sum} - \texttt{numerator}$.
- `Seq.map` turns every numerator into a fraction.

In [ ]:
let fractions_with_sum (sum : int) : Q.t Seq.t =
  Seq.init (sum - 1) (fun i -> i + 1)
  |> Seq.filter (fun numerator -> gcd numerator (sum - numerator) = 1)
  |> Seq.map (fun numerator -> Q.of_ints numerator (sum - numerator))

The function `List.of_seq` converts a finite sequence into a list.

In [ ]:
List.of_seq (fractions_with_sum 5)

The value `rationals` is an infinite *sequence* that enumerates all non-negative rational numbers as follows:

* First, the rational number $\frac{0}{1}$ is generated.
* Second, all rational numbers $\frac{p}{q}$ are computed that satisfy $p > 0$, $p + q = 2$ and $\texttt{gcd}(p, q) = 1$,
* Third, all rational numbers $\frac{p}{q}$ are computed that satisfy $p > 0$, $p + q = 3$ and $\texttt{gcd}(p, q) = 1$,
* $\vdots$

**Implementation:** `Seq.ints 2` is the infinite sequence $2, 3, 4, \cdots$.  The function call `Seq.concat_map f s` applies the function `f` to every element of the sequence `s` and concatenates the resulting sequences.  As sequences are lazy, `Seq.concat_map` only calls `fractions_with_sum` when its elements are needed.  Finally, `Seq.cons x s` puts the element `x` in front of the sequence `s`, and `Q.zero` is the rational number $0$.

In [ ]:
let rationals : Q.t Seq.t =
  Seq.cons Q.zero (Seq.concat_map fractions_with_sum (Seq.ints 2))

In TypeScript, the rational numbers are produced by a generator, which has a method `next` that returns the next rational number.  In OCaml, the function `Seq.to_dispenser` turns the sequence `rationals` into a function `g` of type `unit -> Q.t option`.  Every call `g ()` returns the next rational number `q` as `Some q`.  As the set of rational numbers is infinite, the result is never `None`.

In [ ]:
let g = Seq.to_dispenser rationals

In [ ]:
let v1 = g ()

The toplevel displays a rational number $\frac{p}{q}$ as `p/q`, or as `p` if $q = 1$.

In [ ]:
let v2 = g ()

In [ ]:
g ()

In [ ]:
g ()

The function `gen` takes one argument:
- `cnt` is a natural number.

It prints the first `cnt` rational numbers of the sequence `rationals`, one number per line.  It does not return a meaningful result.

**Implementation:** `Seq.take cnt` takes the first `cnt` elements of the infinite sequence and `Seq.iter` prints each of them.  The format specifier `%5s` prints a string right-aligned in a column of width 5.

In [ ]:
let gen (cnt : int) : unit =
  rationals
  |> Seq.take cnt
  |> Seq.iter (fun q -> Printf.printf "%5s\n" (Q.to_string q))

In [ ]:
gen 100